## 🎲 Exakte Gewinnwahrscheinlichkeiten beim ein- und zwei-Würfel-Fall

Dieses Programm berechnet **exakt** (mittels Bruchrechnung) und **numerisch gerundet** die Gewinnwahrscheinlichkeiten  
$$ P_A(V,W),\ P_B(V,W),\ P_U(V,W) $$  
für zwei vorgegebene **Setzstrategien** $V$ und $W$ für den Fall, dass 
- ein Würfel benutzt wird.
- zwei Würfel benutzt werden.

---

### ✅ Voraussetzungen

- Es werden genau **ein Würfel** und **zwei Würfel** mit benutzerdefinierten Wahrscheinlichkeiten verwendet, die sich jeweils zu $1$ summieren.
- Die Strategien $V$ und $W$ geben an, wie viele Chips auf die jeweiligen Felder gesetzt wurden.

---

### ⚙️ Eingaben

- `V, W`: die beiden Setzstrategien
- `p`: die Wahrscheinlichkeiten für die einzelnen Felder.  
   
---
### 🧩 Modul `PropSetzstrategien.py`

Die Datei `PropSetzstrategien.py` enthält **alle zentralen Funktionen** zur Berechnung:

- Gewinnwahrscheinlichkeiten (`P_A`, `P_B`, `P_U`)
- Strategievergleiche
- Simulationen und Konfidenzintervalle
- Visualisierungen

> 📌 **Wichtig:** Diese Datei muss sich im **Hauptverzeichnis** befinden,  
> damit sie in den Notebooks importiert werden kann (z. B. `import PropSetzstrategien as ps`).

Man kann `PropSetzstrategien.py` als **Funktionssammlung** bezeichnen – sie wird von allen Jupyter-Notebooks gemeinsam genutzt.

---

### ▶️ So geht’s weiter

In der **nächsten Code-Zelle** befindet sich das zugehörige Python-Programm.  
Dort müssen lediglich folgende Eingaben angepasst werden:

- `V`, `W`: Setzstrategien der beiden Spieler (z. B. `V = (3, 2, 1), W=(4,2,0)`)
- `p`: Wahrscheinlichkeiten für die Felder (z. B. `p =[Fraction(1,2), Fraction(1,3), Fraction(1,6)]`)

> 📌 Achte Sie darauf, dass `p`, `V` und `W` gleich lang sind!

---

### ▶️ Ausführen des Programms

- Klicken Sie in die Code-Zelle.
- Drücken Sie `Shift + Enter`, um die Berechnung zu starten.  
  Alternativ können Sie auch auf **▶ Run** oben in der Werkzeugleiste klicken.
- Über **Run → Run all cells** werden alle Zellen auf einmal ausgeführt.


📎 Viel Erfolg beim Experimentieren mit eigenen Strategien!

---

In [18]:
from functools import lru_cache
from fractions import Fraction
import time

# ============== Eingabe ==================
# Beispiel-Strategien und Wahrscheinlichkeiten
V = (3, 2, 1)
W = (4, 2, 0)

# Exakte Wahrscheinlichkeiten als Fraction
p = [Fraction(1,2), Fraction(1,3), Fraction(1,6)]

# ─────────────────────────────────────────────────────────────────────────────
# 1) Ein-Würfel-Fall: rekursiv exakte Gewinnwahrscheinlichkeiten 
# ─────────────────────────────────────────────────────────────────────────────

def make_one_dice_recursion(p_list):
    """
    Baut P_A(V,W) für den Ein-Würfel-Fall:
      P_A(V,W) = sum_{i in Act} (p_i / C) * P_A(V - e_i, W - e_i)
    mit Basisfällen und Normierung C = sum_{i in Act} p_i.
    """
    @lru_cache(maxsize=None)
    def P_A1(V, W):
        V = list(V); W = list(W)
        # Basisfälle
        if sum(V) == 0 and sum(W) > 0:
            return Fraction(1)
        if sum(W) == 0:
            return Fraction(0)
        # aktive Felder
        Act = [i for i in range(len(p_list)) if V[i] + W[i] > 0]
        C = sum(p_list[i] for i in Act)
        x = Fraction(0)
        for i in Act:
            V2 = V.copy(); W2 = W.copy()
            if V2[i] > 0: V2[i] -= 1
            if W2[i] > 0: W2[i] -= 1
            x += Fraction(p_list[i], C) * P_A1(tuple(V2), tuple(W2))
        return x

    return P_A1

# Wrapper für (P_A, P_B, P_U)
def compute_one_dice(V, W, p_list):
    P_A1 = make_one_dice_recursion(p_list)
    PA = P_A1(tuple(V), tuple(W))
    PB = P_A1(tuple(W), tuple(V))
    PU = Fraction(1) - PA - PB
    return PA, PB, PU

# ─────────────────────────────────────────────────────────────────────────────
# 2) Zwei-Würfel-Fall: rekursiv exakte Gewinnwahrscheinlichkeiten
# ─────────────────────────────────────────────────────────────────────────────

def make_two_dice_recursion(p1_list, p2_list):
    """
    Baut P_A(V,W) für den Zwei-Würfel-Fall:
    wir summieren über alle Paare (i,j), entfernen Chips falls möglich
    und eliminieren Selbstschleifen via p_stay.
    """
    @lru_cache(maxsize=None)
    def P_A2(V, W):
        V = list(V); W = list(W)
        # Basisfälle
        if sum(V) == 0 and sum(W) > 0:
            return Fraction(1)
        if sum(W) == 0:
            return Fraction(0)

        X = Fraction(0)
        p_stay = Fraction(0)
        m = len(p1_list)
        for i in range(m):
            for j in range(m):
                pij = p1_list[i] * p2_list[j]
                a_rem = V[i] > 0
                b_rem = W[j] > 0
                if not (a_rem or b_rem):
                    # Selbstschleife: kein Chip entfernt
                    p_stay += pij
                else:
                    V2 = V.copy(); W2 = W.copy()
                    if a_rem: V2[i] -= 1
                    if b_rem: W2[j] -= 1
                    # Absorptionsfälle nach beiden Zügen
                    s2, t2 = sum(V2), sum(W2)
                    if s2 == 0 and t2 > 0:
                        X += pij  # A hat jetzt gewonnen
                    elif t2 == 0:
                        # B hat gewonnen oder Unentschieden → P_A contribution = 0
                        pass
                    else:
                        X += pij * P_A2(tuple(V2), tuple(W2))
        # Eliminierung der Selbstschleifen
        return X / (1 - p_stay)

    return P_A2

# Wrapper für (P_A, P_B, P_U)
def compute_two_dice(V, W, p1_list, p2_list):
    P_A2 = make_two_dice_recursion(p1_list, p2_list)
    PA = P_A2(tuple(V), tuple(W))
    PB = P_A2(tuple(W), tuple(V))
    PU = Fraction(1) - PA - PB
    return PA, PB, PU

# ─────────────────────────────────────────────────────────────────────────────
# 3) Ausgabe
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== EIN-WÜRFEL-FALL (exakt) ===")
t0 = time.perf_counter()
PA1, PB1, PU1 = compute_one_dice(V, W, p)
t1 = time.perf_counter()
print(f"Laufzeit: {t1-t0:.3f}s")
print(f"P_A = {PA1} (≈ {float(PA1):.4f})")
print(f"P_B = {PB1} (≈ {float(PB1):.4f})")
print(f"P_U = {PU1} (≈ {float(PU1):.4f})")

print("\n=== ZWEI-WÜRFEL-FALL (exakt) ===")
t2 = time.perf_counter()
PA2, PB2, PU2 = compute_two_dice(V, W, p, p)
t3 = time.perf_counter()
print(f"Laufzeit: {t3-t2:.3f}s")
print(f"P_A = {PA2} (≈ {float(PA2):.4f})")
print(f"P_B = {PB2} (≈ {float(PB2):.4f})")
print(f"P_U = {PU2} (≈ {float(PU2):.4f})\n")


=== EIN-WÜRFEL-FALL (exakt) ===
Laufzeit: 0.001s
P_A = 1181921/2400000 (≈ 0.4925)
P_B = 65/256 (≈ 0.2539)
P_U = 9511/37500 (≈ 0.2536)

=== ZWEI-WÜRFEL-FALL (exakt) ===
Laufzeit: 0.028s
P_A = 123236774559574168426903996056280400373103/281477364250855970833224885244827648000000 (≈ 0.4378)
P_B = 62849569545570383695296510944165062981331/140738682125427985416612442622413824000000 (≈ 0.4466)
P_U = 6508290120028207003145573460043424332847/56295472850171194166644977048965529600000 (≈ 0.1156)

